In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("employee-handbook.pdf")
documents= loader.load()
documents

In [ ]:
documents[10]

In [3]:
for doc in documents:
    doc.metadata['type'] = "emppolicies"

In [4]:
len(documents)

35

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
len(chunks)

141

In [6]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# e1 = embeddings.embed_query("i love to eat vada pav in mumbai")
# e2= embeddings.embed_query("i love to eat vada pav in mumbai")

In [ ]:
# import numpy as np

# def cosine_similarity(vec1, vec2):
#     return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


# cosine_similarity(e1,e2)


In [12]:
import chromadb
from langchain_chroma import Chroma

chroma_client = chromadb.HttpClient(host="localhost", port=8000)

chromadb= Chroma.from_documents(
            documents=chunks,
            embedding=embeddings, 
            persist_directory="chroma_db"  ,
           # client=chroma_client,
            collection_name="emppoliciesnew"
        )



In [10]:
chromadb = Chroma(
           
            embedding_function=embeddings,
            collection_name="emppoliciesnew"
        )

In [13]:
docs = chromadb.similarity_search_with_score("sick leave", k=3)
docs

[(Document(id='6b155614-3859-46d8-880e-e8ad5807c965', metadata={'type': 'emppolicies', 'xpressprivate': '%%DocumentProcessColors: Cyan Magenta Yellow Black\n%%DocumentCustomColors: (PANTONE 185 C)\n%%+ (PANTONE Blue 072 C)\n%%+ (PANTONE Warm Gray 9 U)\n%%+ (PANTONE 1797 U)\n%%CMYKCustomColor: 0 .91 .76 0 (PANTONE 185 C)\n%%+ 1 .88 0 .05 (PANTONE Blue 072 C)\n%%+ 0 .11 .2 .47 (PANTONE Warm Gray 9 U)\n%%+ 0 1 .99 .04 (PANTONE 1797 U)\n%%EndComments', 'moddate': '2017-03-07T10:21:06-05:00', 'producer': 'QuarkXPress(tm) 6.5', 'page': 15, 'creator': 'QuarkXPress(tm) 6.5', 'source': 'employee-handbook.pdf', 'total_pages': 35, 'page_label': '16', 'creationdate': '2005-11-16T16:29:38+00:00'}, page_content='other health concerns. The company requests that employees provide notification to \ntheir supervisor as soon as practicable when taking time off. Sick days are granted on a \n[paid/unpaid] basis to regular employees. Employees may consult [enter authorized \nperson’s name] regarding the amo

In [14]:
retriever = chromadb.as_retriever(search_type= "mmr", search_kwargs={"k": 3,"fetch_k": 10,"lambda_mult": 0.5})
docs=retriever.invoke("what is the sick leave policy?")
docs


[Document(id='6b155614-3859-46d8-880e-e8ad5807c965', metadata={'page': 15, 'producer': 'QuarkXPress(tm) 6.5', 'moddate': '2017-03-07T10:21:06-05:00', 'type': 'emppolicies', 'page_label': '16', 'xpressprivate': '%%DocumentProcessColors: Cyan Magenta Yellow Black\n%%DocumentCustomColors: (PANTONE 185 C)\n%%+ (PANTONE Blue 072 C)\n%%+ (PANTONE Warm Gray 9 U)\n%%+ (PANTONE 1797 U)\n%%CMYKCustomColor: 0 .91 .76 0 (PANTONE 185 C)\n%%+ 1 .88 0 .05 (PANTONE Blue 072 C)\n%%+ 0 .11 .2 .47 (PANTONE Warm Gray 9 U)\n%%+ 0 1 .99 .04 (PANTONE 1797 U)\n%%EndComments', 'source': 'employee-handbook.pdf', 'creationdate': '2005-11-16T16:29:38+00:00', 'creator': 'QuarkXPress(tm) 6.5', 'total_pages': 35}, page_content='other health concerns. The company requests that employees provide notification to \ntheir supervisor as soon as practicable when taking time off. Sick days are granted on a \n[paid/unpaid] basis to regular employees. Employees may consult [enter authorized \nperson’s name] regarding the amou

In [15]:
from langchain.tools import tool

@tool
def retrieve_employee_policy_data(query):
    """ 

    searches the employee policy data based on the query
     Use this tool when you need to find specific information about:
    - Company policies
    - Employee benefits
    - Working hours and time off
    - Code of conduct
    - Any other employee-related information

    """

    docs=  retriever.invoke(query)

    contents = "\n\n".join([ doc.page_content for doc in docs])

    return contents



In [16]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    model = llm,
    tools = [retrieve_employee_policy_data],
    system_prompt= """You are a helpful assistant that can answer questions about the employee policy data. 
    You are given a question and you need to answer it based on the employee policy data. 
    You can use the retrieve_employee_policy_data tool to retrieve the employee policy data.
    
    Answer the question only based provide data. If you cannot find the answer based on the provided data, say "I don't know"
    
    """
)

result= agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "compare sick leave policy with the maternity leave policy and  work from home policy"}
        ]
    }
)

result['messages']

[HumanMessage(content='compare sick leave policy with the maternity leave policy and  work from home policy', additional_kwargs={}, response_metadata={}, id='734082c9-970b-415e-8e0d-00e887392f36'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 189, 'total_tokens': 262, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0b9898f509', 'id': 'chatcmpl-ETQugvjgBoDm1AxJSuDxu6s4SLp0h', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ed10-685e-7e51-b62a-36b5307968ee-0', tool_calls=[{'name': 'retrieve_employee_policy_data', 'a

In [ ]:
result['messages'][1].tool_calls

In [ ]:
from IPython.display import Image, display,Markdown

Markdown(result['messages'][-1].content)

In [17]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def dynamic_prompt_mw(request: ModelRequest) -> str:

    last_msg= request.state['messages'][-1].content

    docs=retriever.invoke(last_msg)

    contents = "\n\n".join([ doc.page_content for doc in docs])

    system_message = (
        "You are a helpful HR assistant. Use the following context from the employee handbook to answer the question.\n\n"
        "If the answer is not in the context, say \"I don't have that information in the employee handbook.\"\n"
        "Be specific and cite relevant policies when possible.\n\n"
        f"Context:\n{contents}"
    )
    
    return system_message

In [18]:
agent = create_agent(
    model = llm,
    middleware=[dynamic_prompt_mw],
    
)

result =agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "what is the sick leave policy?"}
        ]
    }
)

result

{'messages': [HumanMessage(content='what is the sick leave policy?', additional_kwargs={}, response_metadata={}, id='d6aa06b0-481d-4d6d-9ff3-a037addbc59c'),
  AIMessage(content='The sick leave policy states that sick days are granted on a paid or unpaid basis to regular employees. Employees are required to notify their supervisor as soon as practicable when taking time off. Sick days may not be carried over into the next year, and abuse of this policy may result in disciplinary action. Employees can consult an authorized person regarding the amount of paid sick leave provided each year.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 344, 'total_tokens': 421, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_toke